# FixCheck — SRepair Dataset
Runs FixCheck with codellama on SRepair patches (function-level LLM).
Patches applied using line-number insertion from defects4j-sf.json.

**Before running:** Set runtime to GPU

In [ ]:
from google.colab import files
print('Upload: srepair_prms_results.csv, srepair_data.tar.gz')
uploaded = files.upload()

Upload: srepair_prms_results.csv, srepair_data.tar.gz


Saving srepair_data.tar.gz to srepair_data.tar.gz
Saving srepair_prms_results.csv to srepair_prms_results.csv


In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq openjdk-8-jdk openjdk-11-jdk openjdk-17-jdk \
    git subversion perl cpanminus ant locales wget curl unzip lsof zstd pciutils lshw
locale-gen en_US.UTF-8
echo 'Done'

Preconfiguring packages ...
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../libperl5.38t64_5.38.2-3.2ubuntu0.6_amd64.deb ...
Unpacking libperl5.38t64:amd64 (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2build2.1) ...
Preparing to unpack .../perl_5.38.2-3.2ubuntu0.6_amd64.deb ...
Unpacking perl (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2build2.1) ...
Preparing to unpack .../perl-base_5.38.2-3.2ubuntu0.6_amd64.deb ...
Unpacking perl-base (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2build2.1) ...
Setting up perl-base (5.38.2-3.2ubuntu0.6) ...
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../perl-modules-5.38_5.38.2-3.2ubuntu0.6_all.deb ...
Unpacking perl-modules-5.38 (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2build2.1) ...
Preparing to unpack .../libc6-dev_2.39-0ubuntu8.9_amd64.deb ...
Unpacking libc6-dev:amd64 (2.39-0ubuntu8.9) over (2.39-0ubuntu8.8) ...
Preparing to unpack .../libc-dev-bin_2.39-0ubuntu8.9_amd64.deb .

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh
echo 'Ollama installed'

Ollama installed


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess, time, requests, os

subprocess.run(['pkill', '-f', 'ollama'], capture_output=True)
time.sleep(2)
ollama_env = {
    **os.environ,
    'LD_LIBRARY_PATH': '/usr/lib64-nvidia:/usr/local/cuda-12.8/targets/x86_64-linux/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
    'CUDA_VISIBLE_DEVICES': '0',
}
subprocess.Popen(['ollama', 'serve'],
    stdout=open('/tmp/ollama.log', 'w'),
    stderr=subprocess.STDOUT, env=ollama_env)
time.sleep(6)
try:
    r = requests.get('http://localhost:11434/api/tags')
    print('Ollama running:', r.status_code)
except Exception as e:
    print('ERROR:', e)
with open('/tmp/ollama.log') as f:
    log = f.read()
    if any(x in log for x in ['Tesla','CUDA','A100','L4']):
        print('GPU detected!')
    print(log[-300:])

Ollama running: 200
GPU detected!
i_id=0000:00:03.0 type=discrete total="22.0 GiB" available="21.8 GiB"
time=2026-09-30T18:01:16.421Z level=INFO source=routes.go:2209 msg="vram-based default context" total_vram="22.0 GiB" default_num_ctx=4096
[GIN] 2026/09/30 - 18:01:19 | 200 |     247.712µs |       127.0.0.1 | GET      "/api/tags"



In [ ]:
%%bash
ollama pull codellama
ollama list

NAME                ID              SIZE      MODIFIED               
codellama:latest    8fdf8f752f6e    3.8 GB    Less than a second ago    


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠴ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠇ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 3a43f93b78ec:   0% ▕                  ▏ 765 KB/3.8 GB                  pulling manifest 
pulling 3a43f93b78ec:   3% ▕                  ▏ 111 MB/3.8 GB                  pulling manifest 
pulling 3a43f93b78ec:   7% ▕█                 ▏ 258 MB/3.8 GB                  pulling manifest 
pulling 3a43f93b78ec:   8% ▕█                 ▏ 321 MB/3.8 GB                  pulling manifest 
pulli

In [ ]:
%%bash
update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java
update-alternatives --set javac /usr/lib/jvm/java-17-openjdk-amd64/bin/javac
cd /content
git clone https://github.com/facumolina/fixcheck
cd fixcheck
./gradlew shadowJar
ls build/libs/

update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/java to provide /usr/bin/java (java) in manual mode
update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/javac to provide /usr/bin/javac (javac) in manual mode
...........10%............20%............30%............40%............50%............60%...........70%............80%............90%............100%

Welcome to Gradle 8.0.2!

Here are the highlights of this release:
 - Improvements to the Kotlin DSL
 - Fine-grained parallelism from the first build with configuration cache
 - Configurable Gradle user home cache cleanup

For more details see https://docs.gradle.org/8.0.2/release-notes.html

Starting a Gradle Daemon (subsequent builds will be faster)
> Task :compileJava
> Task :processResources NO-SOURCE
> Task :classes
> Task :shadowJar

BUILD SUCCESSFUL in 22s
2 actionable tasks: 2 executed


Cloning into 'fixcheck'...


In [ ]:
%%bash
pip install -q tokenizers==0.19.1
pip install -q pandas Flask==3.0.3 transformers==4.42.3 scikit-learn
echo 'Done'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.0/321.0 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 474.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 54.3 MB/s eta 0:00:00
Done


  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [ ]:
%%bash
cd /content
git clone https://github.com/rjust/defects4j
update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java
update-alternatives --set javac /usr/lib/jvm/java-8-openjdk-amd64/bin/javac
cpanm --notest DBI String::Interpolate
cd /content/defects4j
perl ./init.sh
echo 'Defects4J init done'

update-alternatives: using /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java to provide /usr/bin/java (java) in manual mode
update-alternatives: using /usr/lib/jvm/java-8-openjdk-amd64/bin/javac to provide /usr/bin/javac (javac) in manual mode
--> Working on DBI
Fetching http://www.cpan.org/authors/id/H/HM/HMBRAND/DBI-1.655.tgz ... OK
Configuring DBI-1.655 ... OK
Building DBI-1.655 ... OK
--> Working on String::Interpolate
Fetching http://www.cpan.org/authors/id/N/NE/NEILB/String-Interpolate-0.33.tar.gz ... OK
Configuring String-Interpolate-0.33 ... OK
==> Found dependencies: PadWalker, Safe::Hole
--> Working on PadWalker
Fetching http://www.cpan.org/authors/id/R/RO/ROBIN/PadWalker-2.5.tar.gz ... OK
Configuring PadWalker-2.5 ... OK
Building PadWalker-2.5 ... OK
--> Working on Safe::Hole
Fetching http://www.cpan.org/authors/id/T/TO/TODDR/Safe-Hole-0.17.tar.gz ... OK
Configuring Safe-Hole-0.17 ... OK
Building Safe-Hole-0.17 ... OK
Building String-Interpolate-0.33 ... OK
4 distributions inst

Cloning into 'defects4j'...
Updating files: 100% (10125/10125), done.
curl: (3) URL using bad/illegal format or missing URL
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 7119k  100 7119k    0     0  3532k      0  0:00:02  0:00:02 --:--:-- 3533k
curl: (3) URL using bad/illegal format or missing URL
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 17.0M  100 17.0M    0     0  18.8M      0 --:--:-- --:--:-- --:--:-- 18.8M
curl: (3) URL using bad/illegal format or missing URL
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --

In [ ]:
%%bash
sed -i '235s/.*/        # die ("Java 11 is required!\\n\\n"); # patched/' /content/defects4j/framework/core/Constants.pm
grep -n 'Java 11' /content/defects4j/framework/core/Constants.pm

235:        # die ("Java 11 is required!\n\n"); # patched


In [ ]:
import os, subprocess
os.environ['PATH'] = '/content/defects4j/framework/bin:/usr/lib/jvm/java-8-openjdk-amd64/jre/bin:' + os.environ['PATH']
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'
r = subprocess.run(['defects4j', 'info', '-p', 'Math'], capture_output=True, text=True)
print(r.stdout[:200])

Summary of configuration for Project: Math
--------------------------------------------------------------------------------
    Script dir: /content/defects4j/framework
      Base dir: /content/defect


In [ ]:
%%bash
mkdir -p /content/srepair_data
tar -xzf /content/srepair_data.tar.gz -C /content/srepair_data
ls /content/srepair_data/
find /content/srepair_data -name '*.json' | head -5

SRepair
srepair_labeled.json
/content/srepair_data/srepair_labeled.json
/content/srepair_data/SRepair/SRepair/dataset/defects4j-sf.json


## Main Loop — Run FixCheck on all SRepair patches

In [ ]:
import os, re, shutil, subprocess, pandas as pd, time, requests, json
from pathlib import Path
from sklearn.metrics import roc_auc_score

os.chdir('/content')

subprocess.run(['pkill', '-f', 'ollama'], capture_output=True)
time.sleep(2)
ollama_env = {
    **os.environ,
    'LD_LIBRARY_PATH': '/usr/lib64-nvidia:/usr/local/cuda-12.8/targets/x86_64-linux/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
    'CUDA_VISIBLE_DEVICES': '0',
}
subprocess.Popen(['ollama', 'serve'],
    stdout=open('/tmp/ollama.log', 'w'),
    stderr=subprocess.STDOUT, env=ollama_env)
time.sleep(6)
try:
    r = requests.get('http://localhost:11434/api/tags')
    print('Ollama running:', r.status_code)
except Exception as e:
    print('Ollama ERROR:', e)

subprocess.run('update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java', shell=True)
subprocess.run('update-alternatives --set javac /usr/lib/jvm/java-8-openjdk-amd64/bin/javac', shell=True)
subprocess.run("sed -i '235s/.*/        # die (\"Java 11 is required!\\\\n\\\\n\"); # patched/' /content/defects4j/framework/core/Constants.pm", shell=True)

JAVA8     = '/usr/lib/jvm/java-8-openjdk-amd64/bin'
JAVA8_JRE = '/usr/lib/jvm/java-8-openjdk-amd64/jre/bin'
JAVA17    = '/usr/lib/jvm/java-17-openjdk-amd64/bin'
D4J       = '/content/defects4j/framework/bin'
BASE      = '/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin'

setup_env = {**os.environ, 'PATH': f'{JAVA8_JRE}:{JAVA8}:{D4J}:{BASE}', 'JAVA_HOME': '/usr/lib/jvm/java-8-openjdk-amd64'}
run_env   = {**os.environ, 'PATH': f'{JAVA17}:{D4J}:{BASE}', 'JAVA_HOME': '/usr/lib/jvm/java-17-openjdk-amd64', 'FIXCHECK': '/content/fixcheck'}

ANT_PROJECTS = {'Chart', 'Closure', 'Time', 'Mockito'}

r = subprocess.run('defects4j info -p Math', shell=True, capture_output=True, text=True, env=setup_env)
print('D4J check:', r.stdout[:50] if r.returncode==0 else r.stderr[:200])

prems_df = pd.read_csv('srepair_prms_results.csv')
prems_df = prems_df[prems_df['status'] == 'OK'].copy().reset_index(drop=True)
prems_df['project'] = prems_df['bug_id'].str.split('-').str[0]
prems_df['bug']     = prems_df['bug_id'].str.split('-').str[1]
prems_df['patch_uid'] = prems_df['bug_id'].str.replace('-', '_') + '_' + prems_df['patch_idx'].astype(str)
print(f'OK patches: {len(prems_df)}')
print(f'Label dist: {prems_df["label"].value_counts().to_dict()}')

labeled = json.load(open('/content/srepair_data/srepair_labeled.json'))
labeled_dict = {f'{item["bug_id"]}_{i}': item for i, item in enumerate(labeled)}
labeled_by_idx = {}
for i, item in enumerate(labeled):
    labeled_by_idx[i] = item
print(f'Labeled patches: {len(labeled)}')

d4j_sf = json.load(open('/content/srepair_data/SRepair/SRepair/dataset/defects4j-sf.json'))
print(f'D4J-SF entries: {len(d4j_sf)}')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_BACKUP = '/content/drive/MyDrive/fixcheck-srepair'
LOCAL_OUTPUT = '/content/fixcheck/fixcheck-output/defects-repairing'
TMP_BASE     = '/content/defects4j_checkouts/base'
CSV_PATH     = f'{DRIVE_BACKUP}/srepair_prems_vs_fixcheck.csv'

os.makedirs(DRIVE_BACKUP, exist_ok=True)
os.makedirs(LOCAL_OUTPUT, exist_ok=True)
os.makedirs(TMP_BASE, exist_ok=True)

restored = 0
for folder in os.listdir(DRIVE_BACKUP):
    src_report = f'{DRIVE_BACKUP}/{folder}/codellama/report.csv'
    if os.path.exists(src_report):
        dst_dir = f'{LOCAL_OUTPUT}/{folder}/codellama'
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(src_report, f'{dst_dir}/report.csv')
        restored += 1
print(f'Restored {restored} report.csv files from Drive')
done = [d for d in os.listdir(LOCAL_OUTPUT) if os.path.exists(f'{LOCAL_OUTPUT}/{d}/codellama/report.csv')]
print(f'Already done: {len(done)}')

def save_csv(results):
    rows = []
    results_dict = {r['patch_uid']: r for r in results}
    for _, row in prems_df.iterrows():
        patch_uid   = row['patch_uid']
        report_path = f'{LOCAL_OUTPUT}/{patch_uid}/codellama/report.csv'
        res = results_dict.get(patch_uid)
        status = res['status'] if res else ('DONE' if os.path.exists(report_path) else 'NOT_RUN')
        if not os.path.exists(report_path):
            rows.append({'patch_uid': patch_uid, 'bug_id': row['bug_id'],
                        'label': int(row['label']), 'prmsc': row['prmsc'],
                        'fixcheck_status': status, 'fixcheck_failing': None, 'fixcheck_pred': None})
            continue
        try:
            report = pd.read_csv(report_path)
            failing = int(report['assertion_failing_prefixes'].sum())
            fixcheck_pred = 0 if failing > 0 else 1
        except:
            failing, fixcheck_pred = None, None
        rows.append({'patch_uid': patch_uid, 'bug_id': row['bug_id'],
                    'label': int(row['label']), 'prmsc': row['prmsc'],
                    'fixcheck_status': 'DONE', 'fixcheck_failing': failing, 'fixcheck_pred': fixcheck_pred})
    df = pd.DataFrame(rows)
    df.to_csv(CSV_PATH, index=False)
    assessable = df[df['fixcheck_pred'].notna()].copy()
    assessable['label'] = pd.to_numeric(assessable['label'], errors='coerce')
    assessable['prmsc'] = pd.to_numeric(assessable['prmsc'], errors='coerce')
    assessable['fixcheck_failing'] = pd.to_numeric(assessable['fixcheck_failing'], errors='coerce')
    assessable = assessable.dropna(subset=['label','prmsc','fixcheck_failing'])
    if len(assessable) > 1 and assessable['label'].nunique() > 1:
        try:
            auc_fc = roc_auc_score(1 - assessable['label'], assessable['fixcheck_failing'])
            auc_pr = roc_auc_score(1 - assessable['label'], 1 - assessable['prmsc'])
            print(f'  [CSV saved — DONE:{len(assessable)} | FixCheck AUC:{auc_fc:.3f} | PReMS AUC:{auc_pr:.3f}]')
        except:
            print(f'  [CSV saved — DONE:{len(assessable)}]')
    return df

def ensure_ollama():
    try:
        requests.get('http://localhost:11434/api/tags', timeout=3)
        return True
    except:
        print('  [Ollama down, restarting...]', end=' ', flush=True)
        subprocess.run(['pkill', '-f', 'ollama'], capture_output=True)
        time.sleep(2)
        subprocess.Popen(['ollama', 'serve'], stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT, env=ollama_env)
        time.sleep(6)
        try:
            requests.get('http://localhost:11434/api/tags', timeout=3)
            print('[Ollama back up]', end=' ', flush=True)
            return True
        except:
            print('[Ollama FAILED]')
            return False

def insert_function(wdir, loc, start, end, patch_code):
    full = os.path.join(wdir, loc)
    if not os.path.exists(full):
        return False
    try:
        with open(full, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
    except:
        return False
    patched, out = False, []
    for idx, line in enumerate(lines):
        if start - 1 <= idx <= end - 1:
            if not patched:
                out.append(patch_code.strip() + '\n')
                patched = True
        else:
            out.append(line)
    if not patched:
        return False
    with open(full, 'w', encoding='utf-8', errors='ignore') as f:
        f.writelines(out)
    return True

def get_trigger(bug_base):
    r = subprocess.run(['defects4j', 'export', '-p', 'tests.trigger'],
                      cwd=bug_base, capture_output=True, text=True, env=setup_env, timeout=60)
    line = r.stdout.strip().split('\n')[0] if r.stdout.strip() else ''
    if '::' in line:
        cls, method = line.split('::', 1)
        return cls.strip(), method.strip()
    return line.strip(), 'test'

def detect_input_class(project_dir, trigger_class, trigger_method):
    class_file = trigger_class.split('.')[-1] + '.java'
    for root, dirs, files in os.walk(project_dir):
        if class_file in files:
            try:
                content = open(os.path.join(root, class_file)).read()
                idx = content.find(f'void {trigger_method}')
                if idx >= 0:
                    brace = content.find('{', idx)
                    if brace >= 0:
                        depth, body = 0, ''
                        for ch in content[brace:]:
                            if ch == '{': depth += 1
                            elif ch == '}': depth -= 1
                            body += ch
                            if depth == 0: break
                        content = body
                if re.search(r'\b\d+\.\d+\b', content) or 'double' in content: return 'double'
                if re.search(r'\b\d+L\b', content) or 'long' in content: return 'long'
                if '"' in content or 'String' in content: return 'java.lang.String'
                if re.search(r'\b\d+\b', content) or 'int' in content: return 'int'
                if 'true' in content or 'false' in content or 'boolean' in content: return 'boolean'
            except: pass
    return 'java.lang.Object'

def get_classpaths(working_dir, project):
    if os.path.isdir(os.path.join(working_dir, 'target/classes')):
        if os.path.isdir(os.path.join(working_dir, 'src/test/java')): tests_src = 'src/test/java'
        elif os.path.isdir(os.path.join(working_dir, 'src/test')): tests_src = 'src/test'
        else: tests_src = 'src/test/java'
        return 'target/classes:target/test-classes', 'target/test-classes', tests_src
    if project == 'Chart': return 'build:build-tests', 'build-tests', 'tests'
    if project == 'Closure': return 'build/classes:build/test', 'build/test', 'test'
    if project == 'Time': return 'build/classes:build/tests', 'build/tests', 'src/test/java'
    if project == 'Mockito': return 'build/classes/main:build/classes/test', 'build/classes/test', 'test'
    return 'target/classes:target/test-classes', 'target/test-classes', 'src/test/java'

def run_fixcheck(patch_uid, working_dir, trigger_class, trigger_method, main_dep, tests_build, tests_src, input_class):
    os.chdir('/content/fixcheck')
    subject_cp = ':'.join([os.path.join(working_dir, d) for d in main_dep.split(':')])
    test_classes_path = os.path.join(working_dir, tests_build)
    target_test_dir = os.path.join(working_dir, tests_src)
    failure_log = os.path.join(working_dir, 'failing_tests')
    with open(failure_log, 'w') as f:
        f.write(f'--- {trigger_class}\n')
        f.write(f'{trigger_class}::{trigger_method}\n')
    props_file = f'/tmp/fixcheck-{patch_uid}.properties'
    with open(props_file, 'w') as f:
        f.write(f'test-classes-path={test_classes_path}\n')
        f.write(f'test-class={trigger_class}\n')
        f.write(f'test-methods={trigger_method}\n')
        f.write(f'test-classes-src={target_test_dir}\n')
        f.write(f'test-failure-trace-log={failure_log}\n')
        f.write(f'inputs-class={input_class}\n')
        f.write(f'number-of-prefixes=20\n')
        f.write(f'assertion-generator=codellama\n')
    full_cp = f'build/libs/fixcheck-all-1.0.0.jar:{subject_cp}:{test_classes_path}'
    cmd = f'/usr/lib/jvm/java-17-openjdk-amd64/bin/java -cp {full_cp} org.imdea.fixcheck.FixCheck -p {props_file}'
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600, env=run_env)
    output_dir = f'fixcheck-output/defects-repairing/{patch_uid}/codellama'
    os.makedirs(output_dir, exist_ok=True)
    for fname in ['report.csv', 'scores-failing-tests.csv', 'log.out']:
        src = f'fixcheck-output/{fname}'
        if os.path.exists(src): subprocess.run(f'mv {src} {output_dir}/', shell=True)
    for dname in ['failing-tests', 'passing-tests', 'non-compiling-tests']:
        src = f'fixcheck-output/{dname}'
        if os.path.exists(src): subprocess.run(f'mv {src} {output_dir}/', shell=True)
    return r.stdout + r.stderr

def backup_report(patch_uid):
    report_src = f'{LOCAL_OUTPUT}/{patch_uid}/codellama/report.csv'
    if os.path.exists(report_src):
        dst_dir = f'{DRIVE_BACKUP}/{patch_uid}/codellama'
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(report_src, f'{dst_dir}/report.csv')

INPUT_TYPES = ['int', 'double', 'long', 'java.lang.String', 'boolean', 'java.lang.Object']

save_csv([])

results = []

for i, row in prems_df.iterrows():
    os.chdir('/content')
    patch_uid  = row['patch_uid']
    project    = row['project']
    bug        = row['bug']
    bug_id     = row['bug_id']
    patch_idx  = int(row['patch_idx'])
    label      = int(row['label'])
    fqcn       = row['fqcn']

    report_file = f'{LOCAL_OUTPUT}/{patch_uid}/codellama/report.csv'
    if os.path.exists(report_file):
        print(f'[{i+1}/{len(prems_df)}] SKIP {patch_uid}')
        results.append({'patch_uid': patch_uid, 'bug_id': bug_id, 'label': label, 'prmsc': row['prmsc'], 'status': 'SKIPPED'})
        continue

    print(f'[{i+1}/{len(prems_df)}] {patch_uid}', end=' ', flush=True)
    ensure_ollama()

    patch_item = labeled_by_idx.get(patch_idx)
    if not patch_item:
        print('NO_PATCH_DATA')
        results.append({'patch_uid': patch_uid, 'bug_id': bug_id, 'label': label, 'prmsc': row['prmsc'], 'status': 'NO_PATCH_DATA'})
        save_csv(results)
        continue

    patch_code = patch_item.get('patch_code', '')
    if not patch_code:
        print('NO_CODE')
        results.append({'patch_uid': patch_uid, 'bug_id': bug_id, 'label': label, 'prmsc': row['prmsc'], 'status': 'NO_CODE'})
        save_csv(results)
        continue

    meta = d4j_sf.get(bug_id)
    if not meta:
        print('NO_META')
        results.append({'patch_uid': patch_uid, 'bug_id': bug_id, 'label': label, 'prmsc': row['prmsc'], 'status': 'NO_META'})
        save_csv(results)
        continue

    loc   = meta['loc']
    start = int(meta['start'])
    end   = int(meta['end'])

    bug_base = os.path.join(TMP_BASE, f'{project}{bug}b')
    if not os.path.isdir(bug_base):
        cmd = f'defects4j checkout -p {project} -v {bug}b -w {bug_base}'
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=300, env=setup_env)
        if r.returncode != 0:
            print('CHECKOUT_FAILED')
            results.append({'patch_uid': patch_uid, 'bug_id': bug_id, 'label': label, 'prmsc': row['prmsc'], 'status': 'CHECKOUT_FAILED'})
            save_csv(results)
            continue
        os.chdir(bug_base)
        subprocess.run('defects4j compile', shell=True, capture_output=True, env=setup_env, timeout=300)

    trigger_class, trigger_method = get_trigger(bug_base)
    if not trigger_class:
        print('NO_TRIGGER')
        results.append({'patch_uid': patch_uid, 'bug_id': bug_id, 'label': label, 'prmsc': row['prmsc'], 'status': 'NO_TRIGGER'})
        save_csv(results)
        continue

    input_class = detect_input_class(bug_base, trigger_class, trigger_method)
    print(f'input={input_class}', end=' ', flush=True)

    working_dir = f'/content/defects4j_checkouts/{patch_uid}/{project}{bug}b'
    os.makedirs(os.path.dirname(working_dir), exist_ok=True)
    if not os.path.isdir(working_dir):
        shutil.copytree(bug_base, working_dir)

    if not insert_function(working_dir, loc, start, end, patch_code):
        print('INSERT_FAILED')
        results.append({'patch_uid': patch_uid, 'bug_id': bug_id, 'label': label, 'prmsc': row['prmsc'], 'status': 'INSERT_FAILED'})
        shutil.rmtree(working_dir, ignore_errors=True)
        save_csv(results)
        continue

    os.chdir(working_dir)
    r_compile = subprocess.run('defects4j compile', shell=True, capture_output=True, text=True, env=setup_env, timeout=300)
    combined_compile = r_compile.stdout + r_compile.stderr
    if 'FAILED' in combined_compile or r_compile.returncode != 0:
        print('COMPILE_FAILED')
        results.append({'patch_uid': patch_uid, 'bug_id': bug_id, 'label': label, 'prmsc': row['prmsc'], 'status': 'COMPILE_FAILED'})
        shutil.rmtree(working_dir, ignore_errors=True)
        save_csv(results)
        continue

    main_dep, tests_build, tests_src = get_classpaths(working_dir, project)
    print('RUN...', end=' ', flush=True)

    final_status = 'UNKNOWN_FAIL'
    for ic in INPUT_TYPES:
        try:
            combined = run_fixcheck(patch_uid, working_dir, trigger_class, trigger_method,
                                   main_dep, tests_build, tests_src, ic)
        except subprocess.TimeoutExpired:
            print('TIMEOUT')
            subprocess.run('pkill -f "fixcheck-all" || true', shell=True)
            final_status = 'TIMEOUT'
            break

        if os.path.exists(report_file):
            print(f'DONE(input={ic})', end=' ', flush=True)
            final_status = 'DONE'
            backup_report(patch_uid)
            print('(backed up)', end=' ', flush=True)
            break
        elif 'No locals of type' in combined:
            continue
        elif 'result" is null' in combined:
            print('TEST_CRASH')
            final_status = 'TEST_CRASH'
            break
        elif 'compile' in combined.lower() and 'failed' in combined.lower():
            print('COMPILE_FAILED')
            final_status = 'COMPILE_FAILED'
            break
        else:
            print('UNKNOWN_FAIL')
            print(f'  tail: {combined[-150:]}')
            final_status = 'UNKNOWN_FAIL'
            break
    else:
        print('NO_LOCALS(all types)')
        final_status = 'NO_LOCALS'

    results.append({'patch_uid': patch_uid, 'bug_id': bug_id, 'label': label, 'prmsc': row['prmsc'], 'status': final_status})
    shutil.rmtree(working_dir, ignore_errors=True)
    save_csv(results)

results_df = pd.DataFrame(results)
print('\n=== Final Summary ===')
print(results_df['status'].value_counts())
final_df = save_csv(results)
from google.colab import files
files.download(CSV_PATH)

Ollama running: 200
D4J check: Summary of configuration for Project: Math
-------
OK patches: 99
Label dist: {0: 75, 1: 24}
Labeled patches: 158
D4J-SF entries: 522
Mounted at /content/drive
Restored 0 report.csv files from Drive
Already done: 0
[1/99] Chart_1_0 input=double RUN... DONE(input=double) (backed up) [2/99] Chart_11_1 input=int RUN... DONE(input=double) (backed up) [3/99] Chart_20_2 input=double RUN... TEST_CRASH
[4/99] Chart_24_3 input=double RUN... DONE(input=double) (backed up) [5/99] Chart_26_4 input=int RUN... DONE(input=int) (backed up)   [CSV saved — DONE:4 | FixCheck AUC:0.333 | PReMS AUC:0.500]
[6/99] Chart_26_5 input=int RUN... DONE(input=int) (backed up)   [CSV saved — DONE:5 | FixCheck AUC:0.333 | PReMS AUC:0.500]
[7/99] Chart_7_6 input=double RUN... DONE(input=double) (backed up)   [CSV saved — DONE:6 | FixCheck AUC:0.250 | PReMS AUC:0.625]
[8/99] Chart_7_7 input=double RUN... DONE(input=double) (backed up)   [CSV saved — DONE:7 | FixCheck AUC:0.458 | PReMS AUC

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files, runtime
files.download(CSV_PATH)